# Explorer les tables NeonDB

Objectif : se connecter à NeonDB depuis le `.env`, vérifier les tables disponibles, puis afficher un aperçu (`head`) de chaque table du projet Foot Predictor.

Le notebook ne modifie aucune donnée : il fait uniquement des lectures SQL.


## 1. Setup

Le notebook cherche une URL de connexion dans `NEON_DATABASE_URL`, `DATABASE_URL` ou `POSTGRES_URL`.


In [19]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / ".env").exists():
    raise FileNotFoundError("Fichier .env introuvable dans foot-predictor.")

SCHEMA = "public"
TABLES = ["competition", "team", "player", "match", "match_team", "lineup", "lineup_audit"]
HEAD_ROWS = 5


## 2. Charger la connexion Neon

On lit le `.env` sans afficher la valeur de connexion.


In [20]:
def load_env_file(path: Path) -> dict[str, str]:
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def sqlalchemy_url(url: str) -> str:
    if url.startswith("postgres://"):
        return url.replace("postgres://", "postgresql+psycopg2://", 1)
    if url.startswith("postgresql://"):
        return url.replace("postgresql://", "postgresql+psycopg2://", 1)
    return url

local_env = load_env_file(PROJECT_ROOT / ".env")
DATABASE_URL = (
    os.environ.get("NEON_DATABASE_URL")
    or os.environ.get("DATABASE_URL")
    or os.environ.get("POSTGRES_URL")
    or local_env.get("NEON_DATABASE_URL")
    or local_env.get("DATABASE_URL")
    or local_env.get("POSTGRES_URL")
    or ""
)

if not DATABASE_URL:
    raise RuntimeError("Ajoute NEON_DATABASE_URL, DATABASE_URL ou POSTGRES_URL dans .env.")

engine = create_engine(sqlalchemy_url(DATABASE_URL), pool_pre_ping=True)
print("Connexion configuree :", bool(DATABASE_URL))


Connexion configuree : True


## 3. Résumé des tables

On vérifie le nombre de lignes et de colonnes avant les aperçus.


In [21]:
summary_rows = []

with engine.begin() as connection:
    for table_name in TABLES:
        row_count = connection.execute(
            text(f'SELECT COUNT(*) FROM "{SCHEMA}"."{table_name}"')
        ).scalar_one()
        column_count = connection.execute(
            text("""
                SELECT COUNT(*)
                FROM information_schema.columns
                WHERE table_schema = :schema
                  AND table_name = :table_name
            """),
            {"schema": SCHEMA, "table_name": table_name},
        ).scalar_one()
        summary_rows.append({"table": table_name, "rows": row_count, "columns": column_count})

tables_summary = pd.DataFrame(summary_rows)
display(tables_summary)


,table,rows,columns
0,competition,45,3
1,team,99,2
2,player,6705,101
3,match,4052,10
4,match_team,8104,7
5,lineup,171684,11
6,lineup_audit,4052,10


## 4. Aperçu de chaque table

Chaque bloc charge seulement quelques lignes avec `LIMIT`, pas toute la table.


In [22]:
heads = {}

for table_name in TABLES:
    query = f'SELECT * FROM "{SCHEMA}"."{table_name}" LIMIT {HEAD_ROWS}'
    heads[table_name] = pd.read_sql(query, engine)
    print(f"\n{table_name}")
    display(heads[table_name])



competition


,competition_id,competition_name,competition_type
0,CGB,CGB,None
1,KLUB,KLUB,None
2,POCP,POCP,None
3,A1,bundesliga,domestic_league
4,L1,bundesliga,domestic_league



team


,team_id,team_name
0,89,1.FC Union Berlin
1,5,AC Milan
2,197,AC Sparta Prague
3,430,ACF Fiorentina
4,2156,AEK Larnaca



player


...

,match_id,team_id,side,score,penalty_score,manager_name,formation
0,3573730,583,away,0,None,Mauricio Pochettino,4-3-3 Attacking
1,3573730,1082,home,1,None,Jocelyn Gourvennec,4-4-2 double 6
2,3575344,27,away,3,None,Julian Nagelsmann,4-2-3-1
3,3575344,16,home,1,None,Marco Rose,4-4-2 Diamond
4,3578075,281,away,0,None,Pep Guardiola,4-3-3 Attacking



lineup


,match_id,team_id,player_id,position_player,lineup_type,is_starting_match,shirt_number,team_captain,minute_start,minute_end,minutes_played
0,3573730,583,99343,Central Midfield,starting_lineup,1,21,0,0,90,90
1,3573730,583,536482,Central Midfield,starting_lineup,1,36,0,0,71,71
2,3573730,583,282041,Centre-Back,starting_lineup,1,3,1,0,81,81
3,3573730,583,228948,Centre-Back,starting_lineup,1,24,0,0,90,90
4,3573730,583,68863,Centre-Forward,starting_lineup,1,9,0,0,90,90



lineup_audit


,match_id,home_starting_count,away_starting_count,home_substitutes_count,away_substitutes_count,home_lineup_rows,away_lineup_rows,complete_starting_xi,has_bench_both_teams,complete_lineup_with_bench
0,3573730,11,11,9,9,20,20,1,1,1
1,3575344,11,11,9,9,20,20,1,1,1
2,3578075,11,11,9,9,20,20,1,1,1
3,3580217,11,11,7,7,18,18,1,1,1
4,3580326,11,11,7,7,18,18,1,1,1


## 5. Exemple de jointure "simple"

Petit aperçu optionnel pour vérifier que `lineup` rejoint bien `match`, `team` et `player`.


In [23]:
lineup_preview_query = f'''
SELECT
    l.match_id,
    m.match_date,
    ht.team_name AS home_team,
    at.team_name AS away_team,
    t.team_name AS lineup_team,
    mt.side,
    mt.score,
    mt.manager_name,
    mt.formation,
    l.lineup_type,
    l.position_player,
    l.minute_start,
    l.minute_end,
    l.minutes_played,
    l.player_id,
    p.player_name,
    p.sofifa_name,
    p.has_sofifa_profile
FROM "{SCHEMA}"."lineup" l
LEFT JOIN "{SCHEMA}"."match" m ON m.match_id = l.match_id
LEFT JOIN "{SCHEMA}"."match_team" mt ON mt.match_id = l.match_id AND mt.team_id = l.team_id
LEFT JOIN "{SCHEMA}"."team" t ON t.team_id = l.team_id
LEFT JOIN "{SCHEMA}"."match_team" hm ON hm.match_id = l.match_id AND hm.side = 'home'
LEFT JOIN "{SCHEMA}"."team" ht ON ht.team_id = hm.team_id
LEFT JOIN "{SCHEMA}"."match_team" am ON am.match_id = l.match_id AND am.side = 'away'
LEFT JOIN "{SCHEMA}"."team" at ON at.team_id = am.team_id
LEFT JOIN "{SCHEMA}"."player" p ON p.player_id = l.player_id
ORDER BY m.match_date DESC, l.match_id, mt.side, l.lineup_type, p.player_name
LIMIT {HEAD_ROWS * 4}
'''

display(pd.read_sql(lineup_preview_query, engine))

,match_id,match_date,home_team,away_team,lineup_team,side,score,manager_name,formation,lineup_type,position_player,minute_start,minute_end,minutes_played,player_id,player_name,sofifa_name,has_sofifa_profile
0,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Right Winger,0.0,90.0,90,536835,Amad Diallo,Amad Diallo,1
1,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Attacking Midfield,0.0,90.0,90,240306,Bruno Fernandes,Bruno Miguel Borges Fernandes,1
2,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Centre-Forward,0.0,74.0,74,413039,Bryan Mbeumo,Bryan Mbeumo,1
3,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Centre-Back,0.0,90.0,90,177907,Harry Maguire,Harry Maguire,1
4,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Defensive Midfield,0.0,90.0,90,820374,Kobbie Mainoo,Kobbie Mainoo,1
5,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Centre-Back,0.0,90.0,90,480762,Lisandro Martínez,Lisandro Martínez,1
6,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Left-Back,0.0,82.0,82,183288,Luke Shaw,Luke Shaw,1
7,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Central Midfield,0.0,74.0,74,346483,Mason Mount,Mason Mount,1
8,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Right-Back,0.0,74.0,74,340456,Noussair Mazraoui,Noussair Mazraoui,1
9,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Left Winger,0.0,62.0,62,926952,Patrick Dorgu,Patrick Dorgu,1


## Check des features descriptives des joueurs

In [24]:
# Création d'un dataframe
players_df = pd.read_sql(f'SELECT * FROM "{SCHEMA}"."player"', engine)

print(players_df.shape)
display(players_df.head())

(6705, 101)


,player_id,player_name,full_name,sofifa_name,player_aliases,date_of_birth,nationality,height_cm,weight_kg,preferred_foot,observed_positions,positions,best_position,club_name,club_league_name,club_position,sofifa_id,has_sofifa_profile,lineup_rows,match_count,team_count,first_match_date,last_match_date,transfermarkt_market_value_eur,transfermarkt_highest_market_value_eur,best_overall,overall_rating,potential,sofifa_market_value_raw,sofifa_wage_raw,skill_moves,weak_foot,international_reputation,body_type,real_face,sofifa_release_clause_raw,acceleration_type,crossing,finishing,heading_accuracy,short_passing,volleys,dribbling,curve,fk_accuracy,long_passing,ball_control,acceleration,sprint_speed,agility,reactions,balance,shot_power,jumping,stamina,strength,long_shots,aggression,interceptions,attack_position,vision,penalties,composure,defensive_awareness,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,pos_ls_rating,pos_st_rating,pos_rs_rating,pos_lw_rating,pos_lf_rating,pos_cf_rating,pos_rf_rating,pos_rw_rating,pos_lam_rating,pos_cam_rating,pos_ram_rating,pos_lm_rating,pos_lcm_rating,pos_cm_rating,pos_rcm_rating,pos_rm_rating,pos_lwb_rating,pos_ldm_rating,pos_cdm_rating,pos_rdm_rating,pos_rwb_rating,pos_lb_rating,pos_lcb_rating,pos_cb_rating,pos_rcb_rating,pos_rb_rating,pos_gk_rating,player_specialities_json,roles_json,playstyles_json
0,889193,AZ Jackson,Aziel Christopher Jackson,Aziel Jackson,AZ Jackson,2001-10-25,United States,175.0,68.0,Right,Left Winger,"CAM,CM,ST",CAM,Jagiellonia Białystok,Ekstraklasa,SUB,262094.0,1,6,6,1,2025-09-24,2025-12-18,NaN,NaN,66.0,64.0,71.0,€1.2M,€3K,3.0,3.0,1.0,Normal (170-185),No,€1.9M,Explosive,52.0,58.0,35.0,65.0,56.0,74.0,40.0,30.0,58.0,65.0,80.0,78.0,86.0,53.0,82.0,64.0,57.0,36.0,58.0,45.0,49.0,30.0,60.0,62.0,56.0,64.0,29.0,40.0,32.0,6.0,12.0,8.0,14.0,6.0,60.0,60.0,60.0,65.0,63.0,63.0,63.0,65.0,64.0,64.0,64.0,63.0,57.0,57.0,57.0,63.0,49.0,48.0,48.0,48.0,49.0,47.0,42.0,42.0,42.0,47.0,14.0,[],"[{""focus"": ""Attack"", ""name"": ""Shadow striker +...",[]
1,1122196,Aaron Bibout,None,None,Aaron Bibout,2004-09-09,Cameroon,193.0,NaN,None,Centre-Forward,None,None,KRC Genk,None,None,NaN,0,15,15,1,2025-08-28,2026-03-19,NaN,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None
2,434207,Aaron Connolly,Aaron Anthony Connolly,Aaron Connolly,Aaron Connolly,2000-01-28,Republic of Ireland,175.0,73.0,Right,"Centre-Forward,Left Winger","ST,LM,CAM,LW",ST,Leyton Orient,League One,CAM,237286.0,1,4,4,1,2021-09-19,2021-10-23,3500000.0,7000000.0,70.0,68.0,72.0,€1.7M,€2K,3.0,3.0,1.0,Normal (170-185),Yes,€3.1M,Controlled explosive,54.0,69.0,65.0,63.0,68.0,68.0,62.0,43.0,45.0,68.0,77.0,73.0,78.0,68.0,78.0,70.0,75.0,67.0,62.0,68.0,68.0,21.0,70.0,58.0,60.0,63.0,25.0,23.0,20.0,12.0,8.0,12.0,13.0,12.0,68.0,68.0,68.0,67.0,68.0,68.0,68.0,67.0,66.0,66.0,66.0,65.0,58.0,58.0,58.0,65.0,49.0,46.0,46.0,46.0,49.0,46.0,43.0,43.0,43.0,46.0,18.0,[],"[{""focus"": ""Attack Support Versatile"", ""name"":...",[]
3,92571,Aaron Cresswell,Aaron William Cresswell,Aaron Cresswell,Aaron Cresswell,1989-12-15,England,175.0,66.0,Left,"Left-Back,Centre-Back","LB,CB,LM",LB,Stoke City,Championship,LB,189615.0,1,97,97,1,2021-08-15,2025-05-18,450000.0,12000000.0,72.0,72.0,72.0,€725K,€18K,2.0,3.0,1.0,Normal (170-185),Yes,€1.4M,Controlled,79.0,53.0,70.0,75.0,48.0,70.0,78.0,74.0,67.0,74.0,52.0,48.0,63.0,72.0,65.0,70.0,74.0,56.0,63.0,66.0,73.0,74.0,60.0,68.0,59.0,71.0,75.0,76.0,75.0,14.0,7.0,9.0,9.0,12.0,64.0,64.0,64.0,66.0,66.0,66.0,66.0,66.0,67.0,67.0,67.0,67.0,69.0,69.0,69.0,67.0,70.0,72.0,72.0,72.0,70.0,70.0,72.0,72.0,72.0,70.0,17.0,[],"[{""focus"": ""Defend Balanced Versatile"", ""name""...",[]
4,591949,Aaron Hickey,Aaron Buchanan Hickey,Aaron Hickey,Aaron 

### Features agrégées SoFIFA

On calcule une version simple des sept groupes visibles sur SoFIFA à partir des caractéristiques détaillées déjà présentes dans `players_df`.

In [25]:
# Agrégation simple des caractéristiques SoFIFA par joueur.
# Chaque groupe est la moyenne des notes détaillées visibles dans la capture SoFIFA.

sofifa_feature_groups = {
    "vit": ["sprint_speed", "acceleration"],
    "tir": ["finishing", "attack_position", "shot_power", "long_shots", "penalties", "volleys"],
    "pas": ["vision", "crossing", "fk_accuracy", "long_passing", "short_passing", "curve"],
    "dri": ["agility", "balance", "reactions", "composure", "ball_control", "dribbling"],
    "defense": ["interceptions", "heading_accuracy", "defensive_awareness", "standing_tackle", "sliding_tackle"],
    "phy": ["jumping", "stamina", "strength", "aggression"],
    "gardien": ["gk_diving", "gk_handling", "gk_kicking", "gk_positioning", "gk_reflexes"],
}

identity_columns = [
    "player_id",
    "player_name",
    "sofifa_name",
    "best_position",
    "observed_positions",
]

sofifa_detail_columns = sorted({
    column
    for columns in sofifa_feature_groups.values()
    for column in columns
})

missing_columns = [
    column
    for column in identity_columns + sofifa_detail_columns
    if column not in players_df.columns
]

if missing_columns:
    raise KeyError(f"Colonnes absentes de players_df: {missing_columns}")

players_with_sofifa_df = players_df.copy()

if "has_sofifa_profile" in players_with_sofifa_df.columns:
    has_sofifa_profile = (
        players_with_sofifa_df["has_sofifa_profile"]
        .fillna(0)
        .astype(str)
        .str.lower()
        .isin(["1", "1.0", "true", "yes"])
    )
    players_with_sofifa_df = players_with_sofifa_df[has_sofifa_profile].copy()

numeric_sofifa_features = players_with_sofifa_df[sofifa_detail_columns].apply(pd.to_numeric, errors="coerce")

for group_name, source_columns in sofifa_feature_groups.items():
    players_with_sofifa_df[group_name] = numeric_sofifa_features[source_columns].mean(axis=1).round(1)

aggregate_columns = list(sofifa_feature_groups.keys())

player_aggregate_features_df = (
    players_with_sofifa_df[identity_columns + aggregate_columns]
    .dropna(subset=aggregate_columns, how="all")
    .sort_values(["best_position", "player_name"], na_position="last")
    .reset_index(drop=True)
)

print(player_aggregate_features_df.shape)
display(player_aggregate_features_df.head(10))
print(player_aggregate_features_df.count())

(5046, 12)


,player_id,player_name,sofifa_name,best_position,observed_positions,vit,tir,pas,dri,defense,phy,gardien
0,889193,AZ Jackson,Aziel Jackson,CAM,Left Winger,79.0,56.5,51.2,70.7,33.2,50.0,9.2
1,646658,Aaron Ramsey,Aaron Ramsey,CAM,Attacking Midfield,72.5,60.5,65.7,70.5,48.6,59.5,11.2
2,724520,Abde Ezzalzouli,Abdessamad Ezzalzouli,CAM,"Right Winger,Left Winger,Left Midfield",88.0,67.2,68.3,78.7,30.6,66.0,7.8
3,340394,Abdelhamid Sabiri,Abdelhamid Sabiri,CAM,"Attacking Midfield,Central Midfield",59.5,66.2,71.3,72.5,59.8,73.8,9.6
4,592400,Abdelkahar Kadri,Abdelkahar Kadri,CAM,Attacking Midfield,72.0,70.2,71.3,77.5,62.8,65.5,9.8
5,340376,Abel Ruiz,Abel Ruiz Ortega,CAM,"Centre-Forward,Left Winger,Second Striker,Left...",74.0,72.7,65.7,73.3,39.4,71.2,8.8
6,881297,Adam Daghim,Adam Daghim,CAM,"Right Winger,Centre-Forward",82.0,61.8,56.2,67.5,51.4,69.0,9.0
7,652061,Adam Karabec,Adam Karabec,CAM,"Attacking Midfield,Right Winger,Left Winger,Le...",69.5,69.5,73.0,72.8,39.6,63.2,11.2
8,43530,Adam Lallana,Adam Lallana,CAM,"Central Midfield,Attacking Midfield,Defensive ...",52.5,70.2,73.7,73.5,62.8,61.0,10.8
9,281906,Adam Vlkanova,Adam Vlkanova,CAM,Attacking Midfield,76.0,62.8,70.0,75.5,34.8,51.8,11.0


player_id             5046
player_name           5046
sofifa_name           5046
best_position         5046
observed_positions    5046
vit                   5046
tir                   5046
pas                   5046
dri                   5046
defense               5046
phy                   5046
gardien               5046
dtype: int64


In [26]:
# Note globale simple hors gardien.
# On exclut `gardien` ici pour garder une note comparable entre joueurs de champ.
columns_note = ["vit", "tir", "pas", "dri", "defense", "phy"]

player_aggregate_features_df["global_note"] = (
    player_aggregate_features_df[columns_note]
    .mean(axis=1)
    .round(1)
)

print(player_aggregate_features_df.shape)
display(player_aggregate_features_df.head(10))

(5046, 13)


,player_id,player_name,sofifa_name,best_position,observed_positions,vit,tir,pas,dri,defense,phy,gardien,global_note
0,889193,AZ Jackson,Aziel Jackson,CAM,Left Winger,79.0,56.5,51.2,70.7,33.2,50.0,9.2,56.8
1,646658,Aaron Ramsey,Aaron Ramsey,CAM,Attacking Midfield,72.5,60.5,65.7,70.5,48.6,59.5,11.2,62.9
2,724520,Abde Ezzalzouli,Abdessamad Ezzalzouli,CAM,"Right Winger,Left Winger,Left Midfield",88.0,67.2,68.3,78.7,30.6,66.0,7.8,66.5
3,340394,Abdelhamid Sabiri,Abdelhamid Sabiri,CAM,"Attacking Midfield,Central Midfield",59.5,66.2,71.3,72.5,59.8,73.8,9.6,67.2
4,592400,Abdelkahar Kadri,Abdelkahar Kadri,CAM,Attacking Midfield,72.0,70.2,71.3,77.5,62.8,65.5,9.8,69.9
5,340376,Abel Ruiz,Abel Ruiz Ortega,CAM,"Centre-Forward,Left Winger,Second Striker,Left...",74.0,72.7,65.7,73.3,39.4,71.2,8.8,66.0
6,881297,Adam Daghim,Adam Daghim,CAM,"Right Winger,Centre-Forward",82.0,61.8,56.2,67.5,51.4,69.0,9.0,64.6
7,652061,Adam Karabec,Adam Karabec,CAM,"Attacking Midfield,Right Winger,Left Winger,Le...",69.5,69.5,73.0,72.8,39.6,63.2,11.2,64.6
8,43530,Adam Lallana,Adam Lallana,CAM,"Central Midfield,Attacking Midfield,Defensive ...",52.5,70.2,73.7,73.5,62.8,61.0,10.8,65.6
9,281906,Adam Vlkanova,Adam Vlkanova,CAM,Attacking Midfield,76.0,62.8,70.0,75.5,34.8,51.8,11.0,61.8


### Jeu d'entrainement minimal

Une ligne par match : `match_id`, les notes globales des 22 titulaires, puis `target_result`.

In [ ]:
# Objectif : construire un dataset minimal pour entrainer un modèle.
# Une ligne = un match.
# Colonnes finales = match_id + 22 notes de titulaires + target_result.

# 1. On récupère les scores par équipe et les joueurs présents dans les lineups.
match_team_df = pd.read_sql(
    f'''
    SELECT match_id, team_id, side, score
    FROM "{SCHEMA}"."match_team"
    ''',
    engine,
)

lineup_df = pd.read_sql(
    f'''
    SELECT match_id, team_id, player_id, is_starting_match
    FROM "{SCHEMA}"."lineup"
    ''',
    engine,
)

# 2. On définit team 1 = équipe à domicile, team 2 = équipe extérieure.
team_1_df = (
    match_team_df[match_team_df["side"] == "home"]
    .rename(columns={"team_id": "team_1_id", "score": "team_1_score"})
    [["match_id", "team_1_id", "team_1_score"]]
)

team_2_df = (
    match_team_df[match_team_df["side"] == "away"]
    .rename(columns={"team_id": "team_2_id", "score": "team_2_score"})
    [["match_id", "team_2_id", "team_2_score"]]
)

# 3. On calcule le résultat du match du point de vue de team 1.
match_result_df = team_1_df.merge(team_2_df, on="match_id", how="inner")
match_result_df["team_1_score"] = pd.to_numeric(match_result_df["team_1_score"], errors="coerce")
match_result_df["team_2_score"] = pd.to_numeric(match_result_df["team_2_score"], errors="coerce")

match_result_df["target_result"] = 0
match_result_df.loc[match_result_df["team_1_score"] > match_result_df["team_2_score"], "target_result"] = 1
match_result_df.loc[match_result_df["team_1_score"] < match_result_df["team_2_score"], "target_result"] = -1

# 4. On récupère uniquement la note globale de chaque joueur.
player_notes_df = player_aggregate_features_df[["player_id", "global_note"]].copy()

# 5. On garde seulement les titulaires et on leur ajoute leur note globale.
lineup_notes_df = (
    lineup_df[lineup_df["is_starting_match"].fillna(0).astype(int) == 1]
    .merge(player_notes_df, on="player_id", how="left")
    .merge(match_result_df[["match_id", "team_1_id", "team_2_id"]], on="match_id", how="inner")
)

# 6. On indique si chaque joueur appartient à team 1 ou team 2.
lineup_notes_df["team_number"] = 2
lineup_notes_df.loc[lineup_notes_df["team_id"] == lineup_notes_df["team_1_id"], "team_number"] = 1

# 7. On numérote simplement les 11 titulaires de chaque équipe.
lineup_notes_df = lineup_notes_df.sort_values(["match_id", "team_number", "player_id"])
lineup_notes_df["player_number"] = lineup_notes_df.groupby(["match_id", "team_number"]).cumcount() + 1
lineup_notes_df = lineup_notes_df[lineup_notes_df["player_number"] <= 11]

# 8. On passe d'un format long à un format large : une ligne par match.
players_notes_wide_df = (
    lineup_notes_df
    .pivot_table(
        index="match_id",
        columns=["team_number", "player_number"],
        values="global_note",
        aggfunc="first",
    )
)

players_notes_wide_df.columns = [
    f"team_{team_number}_player_{player_number}_note"
    for team_number, player_number in players_notes_wide_df.columns
]

players_notes_wide_df = players_notes_wide_df.reset_index()

# 9. Dataset final : match_id + notes joueurs + target_result.
training_dataset_df = (
    match_result_df[["match_id", "target_result"]]
    .merge(players_notes_wide_df, on="match_id", how="inner")
)

# 10. On garde seulement les matchs où les 22 titulaires ont une note disponible.
note_columns = [column for column in training_dataset_df.columns if column.endswith("_note")]
training_dataset_df = training_dataset_df.dropna(subset=note_columns).reset_index(drop=True)

training_dataset_df = training_dataset_df[["match_id"] + note_columns + ["target_result"]]

print(training_dataset_df.shape)
display(training_dataset_df.head())